# 04 - Global Results Comparison for Sparse-View CT

This notebook implements the final global comparison stage of the sparse-view CT pipeline. According to the project specifications, we evaluate and compare the implemented project methods on the exact same degraded test slice and across all requested sparse-view geometries:

1. **Total p-Variation (TpV) Regularization**: the model-based non-convex variational method.
2. **Generalized ResUNet**: the deep learning reconstruction method.

The notebook reports PSNR and SSIM, reconstructed images, error maps, and a final comparison plot across methods.

## 1. Environment Setup and Imports

Mount the Google Drive, install `astra-toolbox` if needed, and import PyTorch, NumPy, Matplotlib, and `IPPy` libraries.

In [ ]:
!pip install astra-toolbox

from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

# Paths setup
PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
PROCESSED_DIR = PROJECT_ROOT / "processed"
RESUNET_WEIGHTS_PATH = PROJECT_ROOT / "weights" / "unet" / "resunet_generalized.pt"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, solvers, utilities
from IPPy.utilities import normalize
from IPPy.utilities.metrics import PSNR, SSIM

# Global Configuration
SEED = 42
IMAGE_SIZE = 256
DETECTOR_SIZE = 256
GEOMETRY = "parallel"
ANGLE_COUNTS = (180, 90, 60, 45)

torch.manual_seed(SEED)
np.random.seed(SEED)
device = utilities.get_device()

print("Device used:", device)
print("Processed data directory:", PROCESSED_DIR)
print("Comparison outputs directory:", OUTPUT_DIR)

## 2. Load the Preprocessed Data Contract (Shared Test Slice)

Load the first test patient file and select the shared central slice to guarantee identical evaluation inputs across all methods and all sparse-view geometries.

In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

# Load the first test patient
test_split = manifest["splits"]["test"]
patient_record = test_split["patients"][0]
patient_path = PROCESSED_DIR / patient_record["path"]
payload = torch.load(patient_path, map_location="cpu")

clean_images = payload["clean"].to(torch.float32)
sinograms = {key: val.to(torch.float32) for key, val in payload["sinograms"].items()}
metadata = payload["metadata"]
patient_id = metadata["patient_id"]

# Select the central slice shared by all methods and all sparse-view settings
slice_idx = clean_images.shape[0] // 2
x_true = clean_images[slice_idx : slice_idx + 1].to(device)
x_true_cpu = x_true.detach().cpu()
angle_keys = [str(n_views) for n_views in ANGLE_COUNTS]

print(f"Loaded Test Patient: {patient_id}")
print(f"Evaluating shared central slice index: {slice_idx}")
print(f"Sparse-view settings under comparison: {', '.join(angle_keys)}")

## 3. FBP Proxy Computation

Compute the FBP image-domain proxy for each sparse-view geometry. This is used only as the degraded input to the ResUNet, not as a separate project method.

In [ ]:
def normalize_finite(image: torch.Tensor) -> torch.Tensor:
    image = torch.nan_to_num(image.float(), nan=0.0, posinf=0.0, neginf=0.0)
    if torch.isclose(image.max(), image.min()):
        return torch.zeros_like(image, dtype=torch.float32)
    return normalize(image).clamp(0.0, 1.0).to(torch.float32)

def compute_metrics(prediction: torch.Tensor, target: torch.Tensor) -> dict[str, float]:
    prediction = torch.clamp(prediction.detach().cpu(), 0.0, 1.0)
    target = torch.clamp(target.detach().cpu(), 0.0, 1.0)
    return {
        "psnr": float(PSNR(prediction, target)),
        "ssim": float(SSIM(prediction, target)),
    }

projectors = {}
comparison_results = {}

for angle_key in angle_keys:
    n_views = int(angle_key)
    K = operators.CTProjector(
        img_shape=(IMAGE_SIZE, IMAGE_SIZE),
        angles=np.linspace(0.0, np.pi, n_views),
        det_size=DETECTOR_SIZE,
        geometry=GEOMETRY,
        force_cpu=True,
    )
    projectors[angle_key] = K

    y_delta = sinograms[angle_key][slice_idx : slice_idx + 1].to(device)
    fbp_solver = solvers.FBP(K)
    print(f"Computing FBP proxy for {angle_key} views...")
    x_fbp, _ = fbp_solver(y_delta, x_true=None, starting_point=None)
    x_fbp_norm = normalize_finite(x_fbp.detach().cpu())

    comparison_results[angle_key] = {
        "fbp_proxy": x_fbp_norm,
        "images": {},
        "metrics": {},
    }

## 4. Method 1: Total p-Variation (TpV) Reconstruction

Run the model-based variational Chambolle-Pock solver from scratch (zeros) on the same test slice and the same saved sinograms used by the other method.

In [ ]:
# Variational Hyperparameters (Calibrated in notebook 02)
lmbda = 0.01          # Regularization weight
p = 0.35              # Sparsity parameter
maxiter = 150         # Maximum iterations

if not (0.1 < p < 0.5):
    raise ValueError(f"Project specifications require 0.1 < p < 0.5. Got p = {p}")

for angle_key in angle_keys:
    y_delta = sinograms[angle_key][slice_idx : slice_idx + 1].to(device)
    tpv_solver = solvers.ChambollePockTpVUnconstrained(projectors[angle_key])

    print(f"Running Chambolle-Pock TpV solver for {angle_key} views...")
    x_tpv, _ = tpv_solver(
        y_delta,
        lmbda=lmbda,
        starting_point=None,
        x_true=x_true,
        maxiter=maxiter,
        tolf=1e-5,
        tolx=1e-5,
        p=p,
        verbose=False,
    )

    x_tpv_norm = normalize_finite(x_tpv.detach().cpu())
    tpv_metrics = compute_metrics(x_tpv_norm, x_true_cpu)
    comparison_results[angle_key]["images"]["TpV (Variational)"] = x_tpv_norm
    comparison_results[angle_key]["metrics"]["TpV (Variational)"] = tpv_metrics
    print(f"  TpV  PSNR: {tpv_metrics['psnr']:.2f} dB | SSIM: {tpv_metrics['ssim']:.4f}")

## 5. Method 2: Deep Learning (Generalized ResUNet)

Load the trained weights of the unified generalized ResUNet and run it on the FBP proxy of the same test slice for every view configuration.

In [ ]:
# Define ResUNet Architecture
class ResidualDoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.shortcut = nn.Conv2d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        h = self.relu(self.conv1(x))
        h = self.conv2(h)
        return self.relu(h + identity)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, block_cls=ResidualDoubleConv):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = block_cls(in_ch, out_ch)
    def forward(self, x):
        return self.block(self.pool(x))

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, block_cls=ResidualDoubleConv):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = block_cls(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.block(x)

class ResidualUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base_ch=32):
        super().__init__()
        self.enc1 = ResidualDoubleConv(in_ch, base_ch)
        self.enc2 = DownBlock(base_ch, 2 * base_ch, block_cls=ResidualDoubleConv)
        self.enc3 = DownBlock(2 * base_ch, 4 * base_ch, block_cls=ResidualDoubleConv)
        self.bottleneck = DownBlock(4 * base_ch, 8 * base_ch, block_cls=ResidualDoubleConv)
        self.dec3 = UpBlock(8 * base_ch, 4 * base_ch, 4 * base_ch, block_cls=ResidualDoubleConv)
        self.dec2 = UpBlock(4 * base_ch, 2 * base_ch, 2 * base_ch, block_cls=ResidualDoubleConv)
        self.dec1 = UpBlock(2 * base_ch, base_ch, base_ch, block_cls=ResidualDoubleConv)
        self.out_conv = nn.Conv2d(base_ch, out_ch, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        h = self.bottleneck(s3)
        h = self.dec3(h, s3)
        h = self.dec2(h, s2)
        h = self.dec1(h, s1)
        return self.out_conv(h)

# Load trained generalized weights
resunet = ResidualUNet(in_ch=1, out_ch=1, base_ch=32).to(device)
checkpoint = torch.load(RESUNET_WEIGHTS_PATH, map_location=device)
resunet.load_state_dict(checkpoint["model_state_dict"])
resunet.eval()

for angle_key in angle_keys:
    print(f"Running ResUNet model for {angle_key} views...")
    fbp_input = comparison_results[angle_key]["fbp_proxy"].to(device)
    with torch.no_grad():
        x_resunet = torch.clamp(resunet(fbp_input), 0.0, 1.0)

    x_resunet_norm = normalize_finite(x_resunet.detach().cpu())
    resunet_metrics = compute_metrics(x_resunet_norm, x_true_cpu)
    comparison_results[angle_key]["images"]["ResUNet (Deep Learning)"] = x_resunet_norm
    comparison_results[angle_key]["metrics"]["ResUNet (Deep Learning)"] = resunet_metrics
    print(f"  ResUNet PSNR: {resunet_metrics['psnr']:.2f} dB | SSIM: {resunet_metrics['ssim']:.4f}")

## 6. Quantitative Results Table

Display a clean ASCII comparative table summarizing PSNR and SSIM for the implemented project methods across all requested sparse-view settings.

In [ ]:
method_order = ["TpV (Variational)", "ResUNet (Deep Learning)"]

print("="*78)
print(f"{ 'VIEWS':<8} | { 'METHOD':<22} | { 'PSNR (dB)':<15} | { 'SSIM':<12}")
print("="*78)
for angle_key in angle_keys:
    for method in method_order:
        metrics = comparison_results[angle_key]["metrics"][method]
        view_label = angle_key if method == method_order[0] else ""
        print(f"{ view_label:<8} | { method:<22} | { metrics['psnr']:<15.4f} | { metrics['ssim']:<12.4f}")
    print("-"*78)
print("="*78)

## 7. Comparative Visual Panels and Quantitative Plot

Plot one final visual comparison panel per view configuration, plus an aggregate PSNR/SSIM plot across the implemented project methods. The figures are saved in the `/outputs/comparison/` directory.

In [ ]:
gt_img = x_true_cpu[0, 0].numpy()

for angle_key in angle_keys:
    tpv_img = comparison_results[angle_key]["images"]["TpV (Variational)"][0, 0].numpy()
    resunet_img = comparison_results[angle_key]["images"]["ResUNet (Deep Learning)"][0, 0].numpy()
    tpv_error = np.abs(tpv_img - gt_img)
    resunet_error = np.abs(resunet_img - gt_img)

    fig, axes = plt.subplots(2, 3, figsize=(12, 8), constrained_layout=True)
    fig.suptitle(f"Project Method Comparison - {angle_key} views - Slice {slice_idx}", fontsize=15)

    axes[0, 0].imshow(gt_img, cmap="gray", vmin=0.0, vmax=1.0)
    axes[0, 0].set_title("Ground Truth")
    axes[0, 0].axis("off")

    for col, (method, image) in enumerate([
        ("TpV (Variational)", tpv_img),
        ("ResUNet (Deep Learning)", resunet_img),
    ], start=1):
        metrics = comparison_results[angle_key]["metrics"][method]
        axes[0, col].imshow(image, cmap="gray", vmin=0.0, vmax=1.0)
        axes[0, col].set_title(f"{method}\nPSNR: {metrics['psnr']:.2f} dB | SSIM: {metrics['ssim']:.4f}")
        axes[0, col].axis("off")

    axes[1, 0].axis("off")

    for col, (title, error_map) in enumerate([
        ("TpV Error |TpV - GT|", tpv_error),
        ("ResUNet Error |ResUNet - GT|", resunet_error),
    ], start=1):
        im_err = axes[1, col].imshow(error_map, cmap="magma")
        axes[1, col].set_title(title)
        axes[1, col].axis("off")
        fig.colorbar(im_err, ax=axes[1, col], shrink=0.8)

    output_fig_path = OUTPUT_DIR / f"comparison_panel_{angle_key}.png"
    fig.savefig(output_fig_path, dpi=150)
    plt.show()
    plt.close(fig)
    print("Saved final comparison panel:", output_fig_path)

colors = ["tab:blue", "tab:green"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
fig.suptitle(f"Project Methods Quantitative Comparison - Test Slice {slice_idx}", fontsize=13)

for method, color in zip(method_order, colors):
    psnr_values = [comparison_results[key]["metrics"][method]["psnr"] for key in angle_keys]
    ssim_values = [comparison_results[key]["metrics"][method]["ssim"] for key in angle_keys]
    axes[0].plot(angle_keys, psnr_values, marker="o", label=method, color=color)
    axes[1].plot(angle_keys, ssim_values, marker="o", label=method, color=color)

axes[0].set_xlabel("Number of views")
axes[0].set_ylabel("PSNR (dB)")
axes[0].set_title("PSNR")
axes[0].legend()

axes[1].set_ylim(0.0, 1.0)
axes[1].set_xlabel("Number of views")
axes[1].set_ylabel("SSIM")
axes[1].set_title("SSIM")
axes[1].legend()

plot_path = OUTPUT_DIR / "project_methods_quantitative_comparison_all_views.png"
fig.savefig(plot_path, dpi=150)
plt.show()
plt.close(fig)

print("Saved global quantitative comparison plot:", plot_path)